In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding verisons!
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)


model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

import torch
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, TrainingArguments
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset, DatasetDict, Dataset
import pandas as pd
import os
from trl import SFTConfig

# 기본 설정값 정의
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
save_model_path = "./trained_model"
data_root = "./data"
batch_size = 2
num_epochs = 3
lora_r = 8
lora_alpha = 24
lora_dropout = 0.1

# 파일 경로 설정
def get_file_paths():
    """Get the appropriate file paths"""
    train_file = "train/train.csv"
    val_file = "test/gemini_test_result.csv"
    save_dir = f"{save_model_path}"
    
    return {
        'train_path': os.path.join(data_root, train_file),
        'val_path': os.path.join(data_root, val_file),
        'save_path': save_dir
    }

# 데이터 로딩 함수
def load_data(paths):
    try:
        print(f"Loading training data from: {paths['train_path']}")
        print(f"Loading validation data from: {paths['val_path']}")
    
        # kwargs 정의가 누락되어 있었습니다 - 아래와 같이 정의합니다
        kwargs = {}
        
        train_data = pd.read_csv(paths['train_path'], **kwargs)
        val_data = pd.read_csv(paths['val_path'], **kwargs)
        
        train_dataset = Dataset.from_pandas(train_data)
        val_dataset = Dataset.from_pandas(val_data)
        
        return DatasetDict({
            'train': train_dataset,
            'validation': val_dataset
        })
    except Exception as e:
        print(f"Error loading data: {e}")
        raise

# 프롬프트 생성 함수
def generate_prompts(row):

    output_texts = []
    for i in range(len(row['text'])):

        text = row['text'][i]
        answer = row['results'][i]

        messages = [{"role":"system","content":"you are a help ful assistant"},
            {"role":"user","content": f"""Please summarize the documentation provided in 3 lines.
    Also, please extract the top five key phrases. See template for the answer format.
    The summary must be written in the same language as the body.
    <template>
    summary
    - summarize 1
    - summarize 2
    - summarize 3

    key phrases
    [key phrase1, key phrase2, key phrase3, key phrase4, key phrase5]
    </template>

    docs:
    {text}"""},{"role": "assistant", "content": f"{answer}"}
        ]
        chat_message = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        output_texts.append(chat_message)
    return output_texts



# 경로 설정
paths = get_file_paths()
os.makedirs(paths['save_path'], exist_ok=True)

print(f"Loading model: {model_name}")
print(f"Model will be saved to: {paths['save_path']}")


print("Loading datasets...")
dataset_dict = load_data(paths)


# 프롬프트 생성 함수
def generate_prompts(examples):

    instructions = examples["text"]
    results       = examples["results"]
    texts = []
    for t, r in zip(instructions, results):

        messages = [{"role":"system","content":"you are a help ful assistant"},
                    {"role":"user","content": f"""Please summarize the documentation provided in 3 lines.
            Also, please extract the top five key phrases. See template for the answer format.
            The summary must be written in the same language as the body.
            <template>
            summary
            - summarize 1
            - summarize 2
            - summarize 3

            key phrases
            [key phrase1, key phrase2, key phrase3, key phrase4, key phrase5]
            </template>

            docs:
            {t}"""},{"role": "assistant", "content": f"{r}"}
                ]
        chat_message = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

        texts.append(chat_message)
    
    return {"text": texts}

# 데이터셋 로드 및 변환
from datasets import load_dataset

dataset_dict = dataset_dict.map(generate_prompts, batched=True)


from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset=dataset_dict['train'],
    eval_dataset=dataset_dict['validation'],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 1000,
        num_train_epochs = 3, # Set this for 1 full training run.
        # train_neum
        learning_rate = 3e-5,
        evaluation_strategy="steps",
        eval_steps=1000,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 500,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
            save_steps=1000,
        save_total_limit=3,
    ),
)

trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

In [16]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 원본 모델과 LoRA 경로
base_model_name = "unsloth/Qwen2.5-1.5B-Instruct"
peft_model_path = "./outputs/checkpoint-3750"  # 학습된 LoRA 모델 경로

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(peft_model_path)

# 기본 모델 로드
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# PEFT 모델 로드
model = PeftModel.from_pretrained(base_model, peft_model_path)

# LoRA 어댑터를 기본 모델에 병합
model = model.merge_and_unload()  # 병합 후 LoRA 제거

# 병합된 모델 저장
merged_model_path = "./Qwen2.5-1.5B-merged"
model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)

('./Qwen2.5-1.5B-merged/tokenizer_config.json',
 './Qwen2.5-1.5B-merged/special_tokens_map.json',
 './Qwen2.5-1.5B-merged/vocab.json',
 './Qwen2.5-1.5B-merged/merges.txt',
 './Qwen2.5-1.5B-merged/added_tokens.json',
 './Qwen2.5-1.5B-merged/tokenizer.json')